In [ ]:
# Import libraries

import os
import json
import random

import numpy as np
import matplotlib.pyplot as plt

import monai
from monai.data import Dataset, DataLoader
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    ConcatItemsd,
    NormalizeIntensityd,
    RandCropByPosNegLabeld,
    Compose,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"MONAI version: {monai.__version__}")

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    "training_dir": os.path.join(BASE_DIR, "data", "brats2020", "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData"),
    "configs": os.path.join(BASE_DIR, "configs"),
    "figures": os.path.join(BASE_DIR, "results", "figures"),
}

for key in ["configs", "figures"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:14s} -> {path}")

In [ ]:
# Build patient list and split into train/val/test
# Split is done at the patient level to avoid data leakage between sets

patient_folders = sorted([
    f for f in os.listdir(PATHS["training_dir"])
    if os.path.isdir(os.path.join(PATHS["training_dir"], f))
])

print(f"Total patients: {len(patient_folders)}")

# Shuffle with fixed seed for reproducibility
shuffled = patient_folders.copy()
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

train_patients = shuffled[:n_train]
val_patients = shuffled[n_train:n_train + n_val]
test_patients = shuffled[n_train + n_val:]

print(f"\nSplit sizes:")
print(f"  Train: {len(train_patients)} ({len(train_patients)/n*100:.1f}%)")
print(f"  Val:   {len(val_patients)} ({len(val_patients)/n*100:.1f}%)")
print(f"  Test:  {len(test_patients)} ({len(test_patients)/n*100:.1f}%)")

# Save the split so it stays fixed across all future notebooks
split_dict = {
    "train": train_patients,
    "val": val_patients,
    "test": test_patients,
}

split_path = os.path.join(PATHS["configs"], "dataset_split.json")
with open(split_path, "w") as f:
    json.dump(split_dict, f, indent=2)

print(f"\nSaved split -> {split_path}")

In [ ]:
# Build MONAI-style data dictionaries: one entry per patient,
# with paths to all 4 modalities and the segmentation mask

def build_data_dicts(patient_list, training_dir):
    data_dicts = []
    modalities = ["t1", "t1ce", "t2", "flair"]

    for pid in patient_list:
        patient_dir = os.path.join(training_dir, pid)
        entry = {
            mod: os.path.join(patient_dir, f"{pid}_{mod}.nii") for mod in modalities
        }
        entry["label"] = os.path.join(patient_dir, f"{pid}_seg.nii")
        entry["patient_id"] = pid
        data_dicts.append(entry)

    return data_dicts


train_dicts = build_data_dicts(train_patients, PATHS["training_dir"])
val_dicts = build_data_dicts(val_patients, PATHS["training_dir"])
test_dicts = build_data_dicts(test_patients, PATHS["training_dir"])

print(f"Train dicts: {len(train_dicts)}")
print(f"Val dicts:   {len(val_dicts)}")
print(f"Test dicts:  {len(test_dicts)}")
print(f"\nExample entry:")
for k, v in train_dicts[0].items():
    print(f"  {k}: {v}")

In [ ]:
# Define the transform pipeline
# This handles loading, multi-modal fusion, normalization, and patch extraction

modalities = ["t1", "t1ce", "t2", "flair"]
patch_size = (96, 96, 96)  # conservative size for 8.6GB VRAM

train_transforms = Compose([
    LoadImaged(keys=modalities + ["label"]),
    EnsureChannelFirstd(keys=modalities + ["label"]),
    ConcatItemsd(keys=modalities, name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=patch_size,
        pos=1,      # 1 part "positive" (contains tumor)
        neg=1,      # 1 part "negative" (random location)
        num_samples=2,  # extract 2 patches per volume per epoch
        image_key="image",
        image_threshold=0,
    ),
])

print("Transform pipeline defined:")
for t in train_transforms.transforms:
    print(f"  - {t.__class__.__name__}")

In [ ]:
# Apply the transform pipeline to one patient and inspect the output patches

sample_dict = train_dicts[0]
result = train_transforms(sample_dict)

print(f"Number of patches extracted: {len(result)}")

for i, patch in enumerate(result):
    image_shape = patch["image"].shape
    label_shape = patch["label"].shape
    tumor_voxels = (patch["label"] > 0).sum().item()
    tumor_pct = tumor_voxels / patch["label"].numel() * 100

    print(f"\nPatch {i}:")
    print(f"  Image shape: {image_shape}")
    print(f"  Label shape: {label_shape}")
    print(f"  Tumor voxels in patch: {tumor_voxels} ({tumor_pct:.2f}% of patch)")

In [ ]:
# Find which depth-slice within this patch actually contains the most tumor

label_patch = patch["label"].numpy()[0]

tumor_per_slice = label_patch.sum(axis=(0, 1))
best_slice = np.argmax(tumor_per_slice)

print(f"Tumor voxels per depth-slice (first 10): {tumor_per_slice[:10].astype(int)}")
print(f"Best slice (most tumor): index {best_slice}, voxels = {int(tumor_per_slice[best_slice])}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(image_patch[3, :, :, best_slice].T, cmap="gray", origin="lower")
axes[0].set_title(f"FLAIR patch — slice {best_slice}")
axes[0].axis("off")

axes[1].imshow(image_patch[3, :, :, best_slice].T, cmap="gray", origin="lower")
masked_label = np.ma.masked_where(label_patch[:, :, best_slice].T == 0, label_patch[:, :, best_slice].T)
axes[1].imshow(masked_label, cmap="autumn", alpha=0.6, origin="lower")
axes[1].set_title(f"FLAIR patch + tumor label — slice {best_slice}")
axes[1].axis("off")

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "patch_samples.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Summary

print("NOTEBOOK 02 COMPLETE")
print("=" * 50)
print(f"Dataset split (patient-level, seed={SEED}):")
print(f"  Train: {len(train_patients)} patients")
print(f"  Val:   {len(val_patients)} patients")
print(f"  Test:  {len(test_patients)} patients")
print(f"  Saved -> configs/dataset_split.json")
print()
print(f"Patch strategy:")
print(f"  Patch size: {patch_size}")
print(f"  Sampling: foreground-biased (pos=1, neg=1) to counter ~1% tumor volume imbalance")
print(f"  Verified: extracted patches contain 2-5% tumor voxels, well above the dataset average")
print()
print("Figures saved:")
for fname in ["patch_samples.png"]:
    path = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Next -> 03_baseline_training.ipynb")
print("  - Build 3D U-Net model (MONAI)")
print("  - Set up Dice + Focal loss")
print("  - Train baseline and log with W&B")